# Лунная База v2.4

## Подключение

In [ ]:
import math, time
from quasar_client import QuasarClient

client = QuasarClient.from_environment()
client.wait_for_simulation_start()

client.reset()
client.configure_incidents(["CybP_02"])
R = "robot.1"


## Модуль безопасности (МБ)

CybP_02

In [ ]:
import base64, hashlib, json
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey, X25519PublicKey
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

b64 = base64.b64decode


def signed(key, sig, text):
    try:
        key.verify(b64(sig), text.encode(), ec.ECDSA(hashes.SHA256()))
        return True
    except Exception:
        return False


def newer(a, b):
    return [int(x) for x in a.split(".")] > [int(x) for x in b.split(".")]


def updates():
    idn = client.get_update_identity()
    if not idn["updatesAvailable"]:
        return
    me = X25519PrivateKey.from_private_bytes(b64(idn["devicePrivateKey"]))
    my_pub = me.public_key().public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw)
    disp = serialization.load_pem_public_key(idn["dispatcherPublicKey"].encode())
    have = {f["module"]: f["version"] for f in idn["firmware"]}

    for _ in range(3):
        ses = client.open_update_session()
        sid, dkey = ses["sessionId"], b64(ses["dispatcherKey"])
        if signed(disp, ses["signature"], sid + "|" + dkey.hex() + "|" + my_pub.hex()):
            break
        client.reject_update(sid, "подпись сеанса не сходится")
        client.journal("error", "CybP_02", "Подменный сеанс отклонён: подпись не сходится", sessionId=sid)

    secret = me.exchange(X25519PublicKey.from_public_bytes(dkey))
    aes = AESGCM(HKDF(hashes.SHA256(), 32, sid.encode(), b"moonbase-update-channel-v1").derive(secret))
    m = client.get_update_manifest(sid)
    man = json.loads(aes.decrypt(b64(m["nonce"]), b64(m["ciphertext"]), (sid + "|manifest").encode()))

    for p in man["packages"]:
        f = client.fetch_update(sid, p["id"])
        try:
            img = aes.decrypt(b64(f["nonce"]), b64(f["ciphertext"]), (sid + "|" + p["id"]).encode())
        except Exception as err:
            print("ошибка расшифровки:", repr(err))
            img = b""
        h = hashlib.sha256(img).hexdigest()
        text = "|".join([p["id"], p["module"], p["version"], p["target"], p["sha256"]])
        if not img:
            why = "образ не расшифровывается"
        elif h != p["sha256"]:
            why = "SHA-256 образа не совпадает с манифестом"
        elif not signed(disp, p["signature"], text):
            why = "подпись пакета не сходится"
        elif p["target"] != idn["hardware"]:
            why = "пакет для другой модели: " + p["target"]
        elif not newer(p["version"], have.get(p["module"], "0.0.0")):
            why = "версия не новее установленной (откат)"
        else:
            why = None
        print(p["id"], p["module"], p["version"], p["target"], why)
        if why:
            client.reject_update(sid, why, p["id"])
            client.journal("error", "CybP_02", "Пакет отклонён: " + why, packageId=p["id"])
        else:
            client.install_update(sid, p["id"], h)
            client.wait_for_firmware()
            have[p["module"]] = p["version"]
            client.journal("info", "CybP_02", "Установлен " + p["module"] + " " + p["version"], packageId=p["id"])
    print("прошивки:", have)


updates()


## Заезд

Задача 1: кнопка. Задача 2: катушка.

In [ ]:
def info(step):
    s = client.get_state()
    r = s["robots"][0]
    g = next(g for g in s["gates"] if g["id"] == "gate.1")
    print(step, round(r["x"], 3), round(r["y"], 3), r["zones"], r["holding"],
          g["state"], s["reactor"]["state"], s["reactor"].get("gatesLockInMs"))
    return s


def go(x, y, v=0.3, tol=0.01):
    client.move_to(R, x, y, speed=v, tolerance=tol, timeout=30)


def open_gate():
    client.open_gate("gate.1")
    client.wait_until(lambda s: next(g for g in s["gates"]
                      if g["id"] == "gate.1")["angle"] > 1.5, timeout=5)


def grab(item):
    client.set_magnet(R, True)
    client.set_velocity(R, 0.03)
    for _ in range(200):
        if client.get_state()["robots"][0]["holding"] == item:
            break
        time.sleep(0.05)
    client.stop(R)


info("старт")
go(0.6, 1.4)
client.rotate_to(R, math.pi / 2, angle_tolerance=0.01)
go(0.6, 2.2)
go(1.007, 2.2)
open_gate()
go(1.007, 2.75)
go(1.104, 2.75)
go(1.104, 2.815, v=0.05, tol=0.003)
for _ in range(4):
    if client.get_state()["reactor"]["state"] == "off":
        break
    client.drive(R, 0.003, speed=0.02)
info("кнопка")
go(1.007, 2.75)
go(1.007, 2.2)
info("снаружи")

client.rotate_to(R, 0, angle_tolerance=0.01)
go(2.975, 2.2)
go(2.975, 2.65)
go(2.87, 2.65)
go(2.87, 2.957, v=0.1, tol=0.004)
client.rotate_to(R, 0, angle_tolerance=0.01)
grab("spool")
info("катушка взята")
go(2.975, 0.6)
go(2.954, 0.21, v=0.1, tol=0.005)
client.set_magnet(R, False)
time.sleep(1)
s = info("катушка")
print(s["spool"], s["charging"]["state"])


In [ ]:
client.close()
